In [ ]:
import numpy as np
from numpy import pi, exp, sqrt, sin, imag, real


V_0 = 0
V_1 = 120
V_2 = 10

V_012 = np.array([
    [V_0],
    [V_1],
    [V_2]
])

def seq_comp(V_012: np.array):
    a = exp(2*pi/3j)
    A = np.array([
        [  1,     1,     1],
        [  1,  a**2,     a],
        [  1,     a,  a**2],
    ])
    V_abc = A @ V_012
    return V_abc

V_abc = seq_comp(V_012)
[V_a, V_b, V_c] = V_abc

t = np.arange(0, 1, 0.001)
f = 60

v_abc_t = np.array([
    [abs(V_a)*sqrt(2)*sin(2*pi*f*t + np.arctan(imag(V_a)/real(V_a)))],
    [abs(V_b)*sqrt(2)*sin(2*pi*f*t + np.arctan(imag(V_b)/real(V_b)))],
    [abs(V_c)*sqrt(2)*sin(2*pi*f*t + np.arctan(imag(V_c)/real(V_c)))]
])

theta_t = 2*pi*f*t

In [ ]:
# 3-Phase Explorer (ASCII): line-to-neutral, line-to-line, Clarke alpha/beta, Park d/q
# Adds a subplot for va, vb, vc (line-to-neutral)
# Uses numpy.deg2rad and two-column sliders to avoid overlap

%matplotlib widget

import numpy as np
from numpy import deg2rad
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# ---- math helpers ----
def phasor(mag, ang_deg):
    return mag * np.exp(1j * deg2rad(ang_deg))

def fortescue_to_phase(V0, V1, V2):
    a = np.exp(1j * 2*np.pi/3)
    Va = V0 + V1 + V2
    Vb = V0 + (a**2)*V1 + a*V2
    Vc = V0 + a*V1 + (a**2)*V2
    return Va, Vb, Vc

def time_domain(V, w, t, phi_ref=0.0):
    return np.sqrt(2) * np.real(V * np.exp(1j*(w*t + phi_ref)))

def clarke_transform(vabc):
    va, vb, vc = vabc
    k = np.sqrt(2/3)
    v_alpha = k * (1*va + (-0.5)*vb + (-0.5)*vc)
    v_beta  = k * (0*va + (np.sqrt(3)/2)*vb + (-np.sqrt(3)/2)*vc)
    v_zero  = k * (1/np.sqrt(2)*va + 1/np.sqrt(2)*vb + 1/np.sqrt(2)*vc)
    return v_alpha, v_beta, v_zero

def park_transform(v_alpha, v_beta, alpha_deg):
    a = deg2rad(alpha_deg)
    cosA, sinA = np.cos(a), np.sin(a)
    v_d =  cosA*v_alpha + sinA*v_beta
    v_q = -sinA*v_alpha + cosA*v_beta
    return v_d, v_q

def make_time_axis(freq_hz, cycles, points_per_cycle):
    T = 1.0 / freq_hz
    N = int(points_per_cycle * cycles)
    return np.linspace(0, cycles*T, N, endpoint=False)

# ---- defaults ----
state = dict(
    V0_mag=0.0,  V0_ang=0.0,
    V1_mag=120.0,V1_ang=0.0,
    V2_mag=0.0,  V2_ang=0.0,
    freq=60.0, cycles=2.0, ppc=800,
    phi_ref_deg=0.0,
    alpha_deg=0.0,
)

# Initial axis for creating line artists
t0 = make_time_axis(state["freq"], state["cycles"], state["ppc"])
z0 = np.zeros_like(t0)

# ---- figure with 4 subplots ----
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
# Some backends ignore setting window title; harmless if so.
try:
    fig.canvas.manager.set_window_title("3-Phase Explorer (ASCII)")
except Exception:
    pass

ax_ln, ax_ll, ax_ab, ax_dq = axes

# line-to-neutral (NEW)
ln_va, = ax_ln.plot(t0, z0, label="va")
ln_vb, = ax_ln.plot(t0, z0, label="vb")
ln_vc, = ax_ln.plot(t0, z0, label="vc")
ax_ln.set_ylabel("Line-neutral (V)")
ax_ln.grid(True); ax_ln.legend(loc="upper right")

# line-to-line
ln_vab, = ax_ll.plot(t0, z0, label="v_ab")
ln_vbc, = ax_ll.plot(t0, z0, label="v_bc")
ln_vca, = ax_ll.plot(t0, z0, label="v_ca")
ax_ll.set_ylabel("Line-line (V)")
ax_ll.grid(True); ax_ll.legend(loc="upper right")

# Clarke alpha/beta
ln_valpha, = ax_ab.plot(t0, z0, label="v_alpha")
ln_vbeta,  = ax_ab.plot(t0, z0, label="v_beta")
ax_ab.set_ylabel("Clarke alpha/beta (V)")
ax_ab.grid(True); ax_ab.legend(loc="upper right")

# Park d/q
ln_vd, = ax_dq.plot(t0, z0, label="v_d")
ln_vq, = ax_dq.plot(t0, z0, label="v_q")
ax_dq.set_ylabel("Park d/q (V)")
ax_dq.set_xlabel("Time (s)")
ax_dq.grid(True); ax_dq.legend(loc="upper right")

# breathing room for 2-column sliders
fig.subplots_adjust(left=0.1, right=0.9, bottom=0.35, top=0.96, hspace=0.28)

# ---- sliders: two columns ----
def add_slider_rect(x, y, label, vmin, vmax, valinit, step=None):
    ax = fig.add_axes([x, y, 0.35, 0.03])
    return Slider(ax, label, vmin, vmax, valinit=valinit, valstep=step)

left_x, right_x = 0.10, 0.60
y_start, dy = 0.28, 0.045

# left column
s_V0_mag = add_slider_rect(left_x,  y_start-0*dy, "V0_mag",     0.0, 300.0, state["V0_mag"], 1.0)
s_V1_mag = add_slider_rect(left_x,  y_start-1*dy, "V1_mag",     0.0, 300.0, state["V1_mag"], 1.0)
s_V2_mag = add_slider_rect(left_x,  y_start-2*dy, "V2_mag",     0.0, 300.0, state["V2_mag"], 1.0)
s_V0_ang = add_slider_rect(left_x,  y_start-3*dy, "V0_ang_deg", -180.0, 180.0, state["V0_ang"], 1.0)
s_V1_ang = add_slider_rect(left_x,  y_start-4*dy, "V1_ang_deg", -180.0, 180.0, state["V1_ang"], 1.0)
s_V2_ang = add_slider_rect(left_x,  y_start-5*dy, "V2_ang_deg", -180.0, 180.0, state["V2_ang"], 1.0)

# right column
s_freq   = add_slider_rect(right_x, y_start-0*dy, "freq_hz",       1.0, 1000.0, state["freq"], 1.0)
s_cycles = add_slider_rect(right_x, y_start-1*dy, "cycles",        1.0,   20.0, state["cycles"], 1.0)
s_ppc    = add_slider_rect(right_x, y_start-2*dy, "pts_per_cycle", 200,  5000,  state["ppc"],   100)
s_phi    = add_slider_rect(right_x, y_start-3*dy, "phi_ref_deg",  -180.0, 180.0, state["phi_ref_deg"], 1.0)
s_alpha  = add_slider_rect(right_x, y_start-4*dy, "alpha_deg",    -180.0, 180.0, state["alpha_deg"],   1.0)

# ---- update logic ----
def update(_=None):
    # read values
    V0 = phasor(s_V0_mag.val, s_V0_ang.val)
    V1 = phasor(s_V1_mag.val, s_V1_ang.val)
    V2 = phasor(s_V2_mag.val, s_V2_ang.val)
    freq   = s_freq.val
    cycles = s_cycles.val
    ppc    = int(s_ppc.val)
    phi    = deg2rad(s_phi.val)
    alpha  = s_alpha.val

    # recompute
    t = make_time_axis(freq, cycles, ppc)
    w = 2*np.pi*freq

    Va, Vb, Vc = fortescue_to_phase(V0, V1, V2)
    va = time_domain(Va, w, t, phi)
    vb = time_domain(Vb, w, t, phi)
    vc = time_domain(Vc, w, t, phi)

    # line-to-neutral (new panel)
    ln_va.set_data(t, va); ln_vb.set_data(t, vb); ln_vc.set_data(t, vc)

    # line-to-line
    vab, vbc, vca = va - vb, vb - vc, vc - va
    ln_vab.set_data(t, vab); ln_vbc.set_data(t, vbc); ln_vca.set_data(t, vca)

    # Clarke and Park
    v_alpha, v_beta, _ = clarke_transform(np.vstack([va, vb, vc]))
    ln_valpha.set_data(t, v_alpha); ln_vbeta.set_data(t, v_beta)
    v_d, v_q = park_transform(v_alpha, v_beta, alpha)
    ln_vd.set_data(t, v_d); ln_vq.set_data(t, v_q)

    # rescale
    for ax in (ax_ln, ax_ll, ax_ab, ax_dq):
        ax.set_xlim(t[0], t[-1])
        ax.relim(); ax.autoscale_view()

    fig.canvas.draw_idle()

# connect sliders
for s in (s_V0_mag, s_V1_mag, s_V2_mag, s_V0_ang, s_V1_ang, s_V2_ang,
          s_freq, s_cycles, s_ppc, s_phi, s_alpha):
    s.on_changed(update)

# initial draw
update()
plt.show()